# Introduction to Stream Processing

**Stream processing** is a way of handling data continuously, as it arrives, instead of waiting to collect it all first. Whereas in batch processing the data is processed in big chunks (batches), in stream processing it is processed event by event, in real time.

```mermaid
flowchart TB

subgraph Stream Processing
    direction TB
    sp1["Rides are continuously<br>streamed into the system"] 
        --> sp2["Each ride is processed<br>immediately"] 
        --> sp3["Dashboard updates<br>in real time"]
end

subgraph Batch Processing
    direction TB
    bp1["Collect all taxi rides for<br>the day"] --> bp2["Run a job at midnight"] 
        --> bp3["Produce a report"]
end
```

In this notebook, you will treat each taxi ride as an event and compare stream thinking with batch thinking. The examples stay small on purpose so the focus stays on windows, thresholds, and event-by-event processing.

**Learning goals:**
- Explain the difference between a batch and a stream
- Treat each taxi ride as an event
- Build a fixed-size event window and summarize it


## Outline

1. Start with a tiny stream of taxi events.
2. Compare batch thinking and stream thinking.
3. Build a simple window.
4. Route events based on a threshold.


In [ ]:
rides = [
    {"ride_id": 1, "total_amount": 14.5, "trip_distance": 1.2},
    {"ride_id": 2, "total_amount": 47.2, "trip_distance": 8.4},
    {"ride_id": 3, "total_amount": 18.8, "trip_distance": 2.3},
    {"ride_id": 4, "total_amount": 63.4, "trip_distance": 12.7},
    {"ride_id": 5, "total_amount": 29.1, "trip_distance": 4.9},
]

rides

## Step 1 - Batch vs stream thinking

In batch processing, we usually wait until we have the full dataset. In stream processing, we react to each event as it arrives.


In [ ]:
batch_total = sum(ride["total_amount"] for ride in rides)
stream_running_total = []
running_total = 0.0

for ride in rides:
    running_total += ride["total_amount"]
    stream_running_total.append(round(running_total, 2))

{"batch_total": round(batch_total, 2), "stream_running_total": stream_running_total}

## Step 2 - Create a fixed-size event window

Windows let us summarize part of a stream without waiting for the entire day or month to finish.


In [ ]:
def fixed_windows(events, window_size):
    return [
        events[index : index + window_size]
        for index in range(0, len(events), window_size)
    ]


windows = fixed_windows(rides, window_size=2)
windows

## Step 3 - Summarize each window

Here we keep the example intentionally small. The important idea is that each window becomes a tiny batch report.


In [ ]:
window_summaries = []
for window in windows:
    summary = {
        "events": len(window),
        "avg_total_amount": round(
            sum(ride["total_amount"] for ride in window) / len(window), 2
        ),
        "avg_trip_distance": round(
            sum(ride["trip_distance"] for ride in window) / len(window), 2
        ),
    }
    window_summaries.append(summary)

window_summaries

## Step 4 - Route events to different outputs

A common stream-processing task is to classify events as they arrive. This notebook uses a simple `total_amount` threshold.


In [ ]:
threshold = 40.0
high_amount_rides = [ride for ride in rides if ride["total_amount"] >= threshold]
low_amount_rides = [ride for ride in rides if ride["total_amount"] < threshold]

{
    "high_amount_rides": high_amount_rides,
    "low_amount_rides": low_amount_rides,
}

## Exercises

- Route rides into three categories, then calculate the average `trip_distance` for each one:
  - economy (`total_amount < 25`)
  - standard (`25 <= total_amount <= 50`)
  - premium (`total_amount > 50`)
- Create windows of three events instead of two. Calculate the min, max and range (max - min) of `total_amount` for each window.
- Describe one business question that is easier to answer with a stream than with a daily batch.


In [ ]:
# TODO: Write your solutions for the exercises above.


<details>
    <summary>Show Reference Solution</summary>

```python
economy_rides = [ride for ride in rides if ride["total_amount"] < 25]
standard_rides = [ride for ride in rides if 25 <= ride["total_amount"] <= 50]
premium_rides = [ride for ride in rides if ride["total_amount"] > 50]

categories = [
    ("economy", economy_rides),
    ("standard", standard_rides),
    ("premium", premium_rides),
]

avg_distances = {}
for name, category_rides in categories:
    avg_distances[name] = {
        "events": len(category_rides),
        "avg_trip_distance": round(
            sum(ride["trip_distance"] for ride in category_rides) / len(category_rides), 2
        ) if category_rides else 0,
    }

avg_distances

# ----------

windows = fixed_windows(rides, window_size=3)

window_stats = {}
for index, window in enumerate(windows, start=1):
    amounts = [ride["total_amount"] for ride in window]
    window_stats[index] = {
        "events": len(window),
        "min_total_amount": min(amounts),
        "max_total_amount": max(amounts),
        "range": max(amounts) - min(amounts),
    }

window_stats
```
</details>
